In [0]:
%run ./_common

In [0]:
import time
from pyspark.sql import functions as F

log = get_logger("opspulse.performance")


def timed(label: str, fn):
    """Run fn(), force execution with .count() if it returns a DataFrame, and log the elapsed time."""
    start = time.time()
    result = fn()
    if hasattr(result, "count"):
        n = result.count()
    else:
        n = result
    elapsed = time.time() - start
    log.info("%s: %.2fs (result=%s)", label, elapsed, n)
    return elapsed, n


def num_partitions(df):
    """RDD API ke bina partition count (serverless-safe)."""
    return df.withColumn("_pid", F.spark_partition_id()).select("_pid").distinct().count()

In [0]:
silver = spark.table(SILVER_TABLE)
print("default partitions:", num_partitions(silver))

def agg_by_repo():
    return silver.groupBy("repo_id").agg(F.count("*").alias("events"))

t_default, _ = timed("no repartition", agg_by_repo)

silver_rep = silver.repartition(8)
print("repartitioned to:", num_partitions(silver_rep))

def agg_by_repo_rep():
    return silver_rep.groupBy("repo_id").agg(F.count("*").alias("events"))

t_rep, _ = timed("repartitioned(8)", agg_by_repo_rep)

print(f"speedup: {t_default / t_rep:.2f}x")

In [0]:
dim = spark.table(DIM_EVENT_TYPE)

def enrich(df):
    return (
        df.join(F.broadcast(dim), on="event_type", how="left")
        .withColumn("category", F.coalesce("category", F.lit("other")))
        .withColumn("weight", F.coalesce("weight", F.lit(1)))
    )

def run_four_aggs(enriched_df):
    a = enriched_df.groupBy("event_date", "event_hour").count()
    b = enriched_df.groupBy("event_date", "repo_id").agg(F.sum("weight").alias("s"))
    c = enriched_df.groupBy("event_date", "actor_login").agg(F.sum("weight").alias("s"))
    d = enriched_df.groupBy("event_date", "event_type").count()
    return a.count() + b.count() + c.count() + d.count()

enriched_nocache = enrich(silver)
t_nocache, _ = timed("no cache, 4 aggregations", lambda: run_four_aggs(enriched_nocache))

cache_supported = True
try:
    enriched_cached = enrich(silver).cache()
    enriched_cached.count()  # materialize the cache
    t_cached, _ = timed("cached, 4 aggregations", lambda: run_four_aggs(enriched_cached))
    print(f"speedup: {t_nocache / t_cached:.2f}x")
    print("is cached:", enriched_cached.is_cached)
except Exception as e:
    cache_supported = False
    t_cached = t_nocache  # summary table ke liye placeholder
    log.warning("cache() not supported on this compute: %s", e)
    print("NOTE: .cache() is not supported on Free Edition serverless compute (PERSIST TABLE not supported).")
    print("This is a documented platform limitation, not a code bug.")

In [0]:
def join_broadcast():
    return silver.join(F.broadcast(dim), on="event_type", how="left")


def join_normal():
    return silver.join(dim, on="event_type", how="left")


t_bcast, _ = timed("broadcast join", lambda: join_broadcast().groupBy("category").count())
t_normal, _ = timed("normal join", lambda: join_normal().groupBy("category").count())
print(f"speedup: {t_normal / t_bcast:.2f}x")

print("--- broadcast join plan ---")
join_broadcast().explain()
print("--- normal join plan ---")
join_normal().explain()

In [0]:
import pandas as pd

summary = pd.DataFrame([
    {"experiment": "repartition(8) vs default", "before_s": round(t_default, 2), "after_s": round(t_rep, 2)},
    {"experiment": "cache vs no cache",         "before_s": round(t_nocache, 2), "after_s": round(t_cached, 2)},
    {"experiment": "broadcast vs normal join",  "before_s": round(t_normal, 2), "after_s": round(t_bcast, 2)},
])
summary["speedup"] = (summary["before_s"] / summary["after_s"]).round(2)
display(spark.createDataFrame(summary))